In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import shapely.wkt
from shapely.geometry import LineString

In [ ]:
# load the cleaned school data
sec_school = pd.read_csv("school_isocrone.csv")

In [ ]:
# List the columns needed to locate and identify each school
required = {'Easting', 'Northing', 'URN', 'EstablishmentName'}

# Find any required columns that are absent from the CSV
missing = required - set(sec_school.columns)

Willis et al., (2004) showed that < 16 years had a speed of 1.53m/s, 16-25 years had 1.55m/s. Their study used a video recording of participants, but did not account for other factors. Pinna and Murrau (2018) accounded for the potential walking in groups, which mirrors the attitude of students, and found 1.04 m/s for young (0–18), 1.00 m/s for adults, 0.99 m/s for middle-aged adults, and 0.84 m/s for older individuals.

In [ ]:
# set walking speeds (m/s) from Pinna & Murrau 2018
walk_speeds_ms = [0.93, 1.04, 1.15]
trip_time_min = [5, 10, 15, 20, 25, 30]
edge_buffer_m = 15

## Download and project the walking network

The network boundary includes the longest possible straight-line walk beyond the outermost school.


In [ ]:
# initialise the GeoDataFrame using British National Grid (meters)
schools_gdf = gpd.GeoDataFrame(
    sec_school,
    geometry=gpd.points_from_xy(sec_school['Easting'], sec_school['Northing']),
    crs='EPSG:27700',
)

In [ ]:
# Calculate the bounding box using the angle properties
min_x, min_y, max_x, max_y = schools_gdf.total_bounds
margin_m = max(walk_speeds_ms) * 60 * max(trip_time_min) + edge_buffer_m

In [ ]:
# Create the bottom-left and top-right corners, with extra space
# around the schools for the longest walking route
corners = gpd.GeoSeries(
    gpd.points_from_xy(
        [min_x - margin_m, max_x + margin_m],  # x coordinates
        [min_y - margin_m, max_y + margin_m]   # y coordinates
    ),
    crs=schools_gdf.crs,                       # EPSG:27700, in metres
).to_crs("EPSG:4326")                          # convert to longitude/latitude

In [ ]:
# Read longitude and latitude from the bottom-left corner
west, south = corners.iloc[0].x, corners.iloc[0].y

# Read longitude and latitude from the top-right corner
east, north = corners.iloc[1].x, corners.iloc[1].y

In EPSG:4326, .x is longitude and .y is latitude. These four values are passed to OSMnx to define the area of walking network to download.

In [ ]:
# Download street network from OpenStreetMap
# Allow up to 300 seconds for an OpenStreetMap request
ox.settings.timeout = 300

# Reuse previously downloaded OpenStreetMap responses when available
ox.settings.use_cache = True

# Download the walking street network within the bounding box
G_raw = ox.graph_from_bbox(
    bbox=(west, south, east, north),
    network_type="walk",
    truncate_by_edge=True
)

In [ ]:
# project the network to British National Grid 
G = ox.project_graph(G_raw, to_crs='EPSG:27700')
schools_projected = schools_gdf.to_crs(G.graph['crs'])

## Generate catchments

Travel times follow the walking network. Polygons buffer complete edges between reached nodes and therefore approximate exact travel-time boundaries.


In [ ]:
# set travel time 
def set_edge_times(graph, speed_ms):
    """
    Calculates the baseline traversal time (in minutes) for every street segment
    (edge) in the network graph based on explicit child walking speeds.
    """
    # Convert meters-per-second to meters-per-minute to align with  
    # temporal isochrone intervals (5 to 30 minutes).
    meters_per_minute = speed_ms * 60
    
    for u, v, k, data in graph.edges(keys=True, data=True):
        # Calculate time cost and inject it into the edge attribute metadata.
        # NetworkX 'ego_graph' will later use this 'time' key as its travel weight.
        data["time"] = data["length"] / meters_per_minute

In [ ]:
# spatial geometry conversion
def isochrone_geometry(graph, center_node, trip_time_min, buffer_m=15):
    """
    Extracts reachable network components for a specific time threshold 
    and constructs a precise edge-buffered polygon catchment area.
    """
    # slice the network graph using Dijkstra-based tracking along the custom 'time' weights.
    subgraph = nx.ego_graph(graph, center_node, radius=trip_time_min, distance="time")
    
    # extract explicit geometries from the reached streets to account for true road curvature.
    edge_lines = []
    for u, v, data in subgraph.edges(data=True):
        if "geometry" in data:
            edge_lines.append(data["geometry"])
        else:
            # draw straight lines between intersection coordinates if explicit shapes are missing.
            # Safe because coordinates 'x' and 'y' are verified to be metric (meters) from the projection.
            edge_lines.append(LineString([
                (subgraph.nodes[u]["x"], subgraph.nodes[u]["y"]),
                (subgraph.nodes[v]["x"], subgraph.nodes[v]["y"]),
            ]))

    # An isolated node has no lines to buffer.
    if not edge_lines:
        return None, len(subgraph.nodes)
    # process the lines using GeoPandas within the metric coordinate space
    edges_gdf = gpd.GeoSeries(edge_lines, crs=graph.graph["crs"])
    
    # generate the metric buffer and dissolve intersecting paths into a unified polygon footprint
    buffered = edges_gdf.buffer(buffer_m)
    unioned = buffered.union_all()
    
    # project the metric polygon back to global degrees (EPSG:4326 WGS84) for data storage uniformity.
    unioned_wgs84 = gpd.GeoSeries([unioned], crs=graph.graph["crs"]).to_crs("EPSG:4326").iloc[0]

    return unioned_wgs84, len(subgraph.nodes)

In [ ]:
# loop for each speed and trip times
records = []

for speed_ms in walk_speeds_ms:
    set_edge_times(G, speed_ms)
    
    for _, row in schools_projected.iterrows():
        # Original field osm_id contains the school URN; retained for compatibility.
        sch_id = row["URN"]
        
        # Snap the school's metric coordinates directly to the graph
        proj_x = row.geometry.x
        proj_y = row.geometry.y
        center_node = ox.distance.nearest_nodes(G, X=proj_x, Y=proj_y)

        # Generate a catchment for each walking-time threshold, in ascending order
        for trip_time in sorted(trip_time_min):
            poly_geom, n_nodes = isochrone_geometry(G, center_node, trip_time, buffer_m=edge_buffer_m)

            # Skip thresholds for which no usable polygon was generated
            if poly_geom is None or poly_geom.is_empty:
                continue

            # Store the school, walking assumptions, and polygon for this catchment
            records.append({
                "osm_id": sch_id,
                "school_name": row["EstablishmentName"],
                "walk_speed_ms": speed_ms,
                "trip_time_min": trip_time,
                "band_label": f"0-{trip_time}",
                "n_nodes_reached": n_nodes,
                "geometry_wkt": poly_geom.wkt,
            })

In [ ]:
# isochrone_geometry() has already converted the polygons to EPSG:4326.
isochrones_gdf = gpd.GeoDataFrame(
    isochrone_df.drop(columns='geometry_wkt'),
    geometry=isochrone_df['geometry_wkt'].apply(shapely.wkt.loads),
    crs='EPSG:4326',
)

In [ ]:
isochrones_gdf.head()

In [ ]:
# save the data
isochrones_gdf.to_file('school_catchments.geojson', driver='GeoJSON')